# Neural Collaborative Filtering -- Beyond Dot Products

Matrix factorization predicts ratings with a dot product: U[u] . V[i]. The dot product is a
**linear** operation. It can model "this user likes action and this film is action-heavy," but
it cannot model "this user likes comedies, but only short ones" -- that requires combining
genre AND runtime in a non-additive way.

**Neural collaborative filtering (NCF)** replaces the dot product with an MLP. User and item
embeddings are concatenated and passed through hidden layers that can learn arbitrary
interactions.

This notebook uses PyTorch. The four-step training loop (predict, score, backward, update)
is the same as always.

| Step | What we build |
|------|--------------|
| 1 | Why the dot product is limited |
| 2 | NCF: embed -> concatenate -> MLP |
| 3 | Two-tower architecture for fast serving |
| 4 | Adding content features to address cold start |

In [ ]:
import random, math, matplotlib.pyplot as plt
import torch
import torch.nn as nn
%matplotlib inline

def make_ratings(n_users=80, n_items=120, n_factors=4, density=0.07, seed=42):
    random.seed(seed)
    U = [[random.gauss(0,1) for _ in range(n_factors)] for _ in range(n_users)]
    V = [[random.gauss(0,1) for _ in range(n_factors)] for _ in range(n_items)]
    bu = [random.gauss(0, 0.3) for _ in range(n_users)]
    bv = [random.gauss(0, 0.3) for _ in range(n_items)]
    mu = 3.5
    ratings = []
    for u in range(n_users):
        for v in range(n_items):
            if random.random() < density:
                r = mu + bu[u] + bv[v] + sum(U[u][k]*V[v][k] for k in range(n_factors))
                r = max(1.0, min(5.0, r + random.gauss(0, 0.3)))
                ratings.append((u, v, round(r)))
    return ratings, n_users, n_items

def train_val_split(ratings, val_frac=0.2, seed=42):
    random.seed(seed)
    shuffled = list(ratings)
    random.shuffle(shuffled)
    split = int(len(shuffled) * (1 - val_frac))
    return shuffled[:split], shuffled[split:]

def rmse_torch(model, data, extra_fn=None):
    model.eval()
    with torch.no_grad():
        us = torch.tensor([u for u, i, r in data])
        it = torch.tensor([i for u, i, r in data])
        rs = torch.tensor([float(r) for u, i, r in data])
        preds = model(us, it) if extra_fn is None else extra_fn(model, us, it)
        return math.sqrt(((preds - rs) ** 2).mean().item())

ratings, N_USERS, N_ITEMS = make_ratings()
train, val = train_val_split(ratings)
print(f'{len(train)} train ratings, {len(val)} val ratings')

## Step 1: The Limitation of the Dot Product

The dot product U[u] . V[i] computes one number: the summed product of corresponding
dimensions. Each dimension contributes independently. There is no mechanism for one dimension
to modulate another.

Consider a preference that depends on the *combination* of two factors:
- A user likes comedies (factor 0 = 1) AND short films (factor 1 = 1) -> high rating
- A user likes comedies (factor 0 = 1) AND long films (factor 1 = -1) -> low rating

For the dot product, factor 0 must get a fixed weight and factor 1 must get a fixed weight.
It cannot express "factor 0 matters differently depending on factor 1."

An MLP can: the first hidden layer computes combinations of dimensions (e.g., factor_0 *
factor_1), later layers detect arbitrary conjunctions.

### The NCF architecture

```
r_hat(u, i) = MLP( concat(U[u], V[i]) )
```

U[u] and V[i] are each `emb_dim` numbers; the MLP receives `2 * emb_dim` numbers as input
and outputs one predicted rating. The MLP can learn any function of the combined embedding,
including non-linear interactions the dot product cannot express.

### nn.Embedding

PyTorch's `nn.Embedding(n, d)` is a lookup table of shape (n, d). Given a batch of integer
IDs, it returns the corresponding rows. The embedding weights are parameters: they are
initialised randomly and updated by backprop exactly like `nn.Linear` weights.

This is the same idea as the `C` matrix in the embeddings notebook -- a lookup table indexed
by token ID. Here the IDs are user IDs and item IDs instead of token IDs.

## Step 2: Building the NCF Model in PyTorch

The model in three lines of logic:

1. Look up user embedding -> `emb_dim` numbers
2. Look up item embedding -> `emb_dim` numbers
3. Concatenate and pass through an MLP -> one predicted rating

No softmax or sigmoid. This is a regression task: predicting a rating on [1, 5].

In [ ]:
class NCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=16, hidden=[64, 32]):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        layers = []
        in_dim = emb_dim * 2
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU()]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, users, items):
        u = self.user_emb(users)             # (batch, emb_dim)
        v = self.item_emb(items)             # (batch, emb_dim)
        x = torch.cat([u, v], dim=1)         # (batch, 2*emb_dim)
        return self.mlp(x).squeeze(1)        # (batch,)

torch.manual_seed(42)
ncf = NCF(N_USERS, N_ITEMS)
n_params = sum(p.numel() for p in ncf.parameters())
print(f'NCF parameters: {n_params}')
print(ncf)

In [ ]:
def train_model(model, train_data, val_data, epochs=20, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    train_rmse_hist, val_rmse_hist = [], []

    us_tr = torch.tensor([u for u, i, r in train_data])
    it_tr = torch.tensor([i for u, i, r in train_data])
    rs_tr = torch.tensor([float(r) for u, i, r in train_data])

    for epoch in range(epochs):
        model.train()
        # 1. Predict
        preds = model(us_tr, it_tr)
        # 2. Score
        loss = loss_fn(preds, rs_tr)
        # 3. Assign blame
        optimizer.zero_grad()
        loss.backward()
        # 4. Nudge
        optimizer.step()

        tr = rmse_torch(model, train_data)
        va = rmse_torch(model, val_data)
        train_rmse_hist.append(tr)
        val_rmse_hist.append(va)
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f'Epoch {epoch:2d}  train RMSE={tr:.4f}  val RMSE={va:.4f}')

    return train_rmse_hist, val_rmse_hist

torch.manual_seed(42)
ncf = NCF(N_USERS, N_ITEMS)
tr_ncf, va_ncf = train_model(ncf, train, val)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(tr_ncf, label='NCF train RMSE')
ax.plot(va_ncf, label='NCF val RMSE')
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
ax.set_title('NCF Training')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 3: The Two-Tower Architecture

NCF entangles user and item representations from the first hidden layer: both embeddings
are concatenated and processed together. At inference time, every candidate user-item pair
requires a fresh forward pass through the full model.

With 80 million users and 10 million items, that is 800 trillion forward passes to rank all
items for all users. Not feasible.

### The two-tower idea

Keep user and item encoders completely separate -- one **tower** per side. Only combine at
the final step via a dot product:

```
user_vec = UserTower(u)      # a deep encoding of the user
item_vec = ItemTower(i)      # a deep encoding of the item
score    = user_vec . item_vec
```

**Why this is fast at serving time:**
- Item vectors are precomputed once for the entire catalogue.
- At request time, run only the user tower (one forward pass per user).
- Find the K nearest item vectors using approximate nearest-neighbour (ANN) search.
- Total cost: O(UserTower) + O(ANN) -- independent of catalogue size.

YouTube and Pinterest use exactly this architecture for first-stage retrieval.

The trade-off: a dot product is less expressive than an MLP over the concatenated vector.
Two-tower retrieves a coarse set of candidates; a heavier re-ranking model then refines them.

In [ ]:
class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=16, hidden=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.user_fc  = nn.Sequential(
            nn.Linear(emb_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.item_fc  = nn.Sequential(
            nn.Linear(emb_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden)
        )

    def forward(self, users, items):
        u = self.user_fc(self.user_emb(users))   # (batch, hidden)
        v = self.item_fc(self.item_emb(items))   # (batch, hidden)
        return (u * v).sum(dim=1)                 # dot product, shape (batch,)

torch.manual_seed(42)
tt = TwoTower(N_USERS, N_ITEMS)
print(f'Two-Tower parameters: {sum(p.numel() for p in tt.parameters())}')

In [ ]:
torch.manual_seed(42)
tt = TwoTower(N_USERS, N_ITEMS)
tr_tt, va_tt = train_model(tt, train, val)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(tr_ncf, linestyle='--', label='NCF train',      alpha=0.7)
ax.plot(va_ncf, linestyle='--', label='NCF val',        alpha=0.7)
ax.plot(tr_tt,               label='Two-Tower train')
ax.plot(va_tt,               label='Two-Tower val')
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
ax.set_title('Two-Tower vs NCF')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'NCF final val RMSE:       {va_ncf[-1]:.4f}')
print(f'Two-Tower final val RMSE: {va_tt[-1]:.4f}')

## Step 4: Adding Content Features (Cold Start)

Both NCF and two-tower inherit matrix factorization's cold start problem: a new item with
no ratings has a randomly initialised embedding and cannot be recommended meaningfully.

The fix: let the item tower consume **content features** -- information known about an item
before any user rates it. In a real system: genre, release year, runtime, director, description
embedding. The item tower takes both the item ID (for the learned collaborative signal) and
the content features (for the structural signal) as input.

**What this buys us:** a new item with zero ratings can still get a reasonable score from
its genre or description. As ratings accumulate, the learned embedding takes over.

### Synthetic genre features

Our synthetic data has 4 latent factors. We simulate genre as a one-hot vector of length 4,
where item i belongs to genre (i % 4). In a real system this would be an actual genre label
or a richer feature vector.

### Content-aware item tower

```
item_input = concat(ItemEmbedding(i), ContentEncoder(genre))
item_vec   = ItemMLP(item_input)
```

For a new item (ID embedding is random noise), ContentEncoder(genre) still provides a
meaningful signal. For a well-rated item, both signals contribute.

In [ ]:
# Assign each item a genre (0-3) based on its index.
N_GENRES = 4
item_genre = [i % N_GENRES for i in range(N_ITEMS)]

def genre_onehot(item_id):
    v = [0.0] * N_GENRES
    v[item_genre[item_id]] = 1.0
    return v

print('Genre assignments (first 8 items):', [item_genre[i] for i in range(8)])
print('One-hot for item 0:', genre_onehot(0))

In [ ]:
class ContentAwareTwoTower(nn.Module):
    def __init__(self, n_users, n_items, n_genres, emb_dim=16, hidden=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.user_fc  = nn.Sequential(
            nn.Linear(emb_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.item_emb   = nn.Embedding(n_items, emb_dim)
        self.content_fc = nn.Linear(n_genres, emb_dim)  # encodes genre features
        # Item tower receives: learned embedding + genre encoding, concatenated
        self.item_fc    = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden)
        )

    def forward(self, users, items, genres):
        u = self.user_fc(self.user_emb(users))

        id_emb    = self.item_emb(items)                     # from ratings
        genre_emb = torch.relu(self.content_fc(genres))      # from content
        item_in   = torch.cat([id_emb, genre_emb], dim=1)
        v         = self.item_fc(item_in)

        return (u * v).sum(dim=1)

torch.manual_seed(42)
cat_model = ContentAwareTwoTower(N_USERS, N_ITEMS, N_GENRES)
print(f'Content-aware parameters: {sum(p.numel() for p in cat_model.parameters())}')

In [ ]:
def train_content_model(model, train_data, val_data, epochs=20, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    train_rmse_hist, val_rmse_hist = [], []

    def to_tensors(data):
        us = torch.tensor([u for u, i, r in data])
        it = torch.tensor([i for u, i, r in data])
        ge = torch.tensor([genre_onehot(i) for _, i, _ in data])
        rs = torch.tensor([float(r) for u, i, r in data])
        return us, it, ge, rs

    us_tr, it_tr, ge_tr, rs_tr = to_tensors(train_data)
    us_va, it_va, ge_va, rs_va = to_tensors(val_data)

    for epoch in range(epochs):
        model.train()
        preds = model(us_tr, it_tr, ge_tr)
        loss  = loss_fn(preds, rs_tr)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            tr = math.sqrt(loss_fn(model(us_tr, it_tr, ge_tr), rs_tr).item())
            va = math.sqrt(loss_fn(model(us_va, it_va, ge_va), rs_va).item())
        train_rmse_hist.append(tr)
        val_rmse_hist.append(va)
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f'Epoch {epoch:2d}  train RMSE={tr:.4f}  val RMSE={va:.4f}')

    return train_rmse_hist, val_rmse_hist

torch.manual_seed(42)
cat_model = ContentAwareTwoTower(N_USERS, N_ITEMS, N_GENRES)
tr_cat, va_cat = train_content_model(cat_model, train, val)

In [ ]:
# Cold-start demonstration: personalised predictions for a 'new' genre-0 item.
# We use item 0 (which has the same genre one-hot) as a proxy.
# In a real system, the item ID embedding for a brand-new item is random noise;
# the content encoder (genre) carries all the signal.

cat_model.eval()
genre0_onehot = torch.tensor([[1.0, 0.0, 0.0, 0.0]])  # genre 0

with torch.no_grad():
    print('Predicted ratings for a new genre-0 item, different users:')
    for uid in range(5):
        u_t = torch.tensor([uid])
        i_t = torch.tensor([0])   # proxy for any genre-0 item
        pred = cat_model(u_t, i_t, genre0_onehot)
        print(f'  User {uid}: predicted {pred.item():.2f}')

print()
print('Predictions differ across users -- personalisation works via the user tower.')
print('Content features give the item tower a baseline even before any ratings.')

## Summary

| Property | Matrix Factorization | NCF | Two-Tower | Content-Aware Two-Tower |
|----------|---------------------|-----|-----------|------------------------|
| Model | Dot product | MLP over concat | MLP towers + dot | MLP towers + content + dot |
| Expressivity | Linear | Non-linear | Non-linear | Non-linear |
| Serving cost (inference) | O(1) | O(1) per pair | O(user) + ANN | O(user) + ANN |
| Cold start (new items) | Fails | Fails | Fails | Partial -- content helps |
| Cold start (new users) | Fails | Fails | Fails | Fails |
| Interpretability | Moderate | Low | Low | Low |

### What we learned

- **The dot product is linear.** NCF replaces it with an MLP to capture arbitrary user-item interactions.
- **`nn.Embedding` is a lookup table** trained by backprop -- identical to the word embeddings in the
  GPT track, indexed by user or item ID instead of token ID.
- **Two-tower separates encoding from scoring.** Item vectors are precomputed; at serving time only
  the user tower runs. This is the standard architecture for retrieval at scale.
- **Content features address cold start partially.** A new item's genre gives the model a baseline
  before any ratings arrive. As ratings accumulate, the learned embedding takes over.
- **The four-step loop is always the same.** Whether the model is 60 lines of pure Python (notebook 3)
  or a PyTorch MLP, training is: predict -> score -> backward -> update.

## Your Turn

**1. Embedding dimension.** Retrain NCF with `emb_dim=4` and `emb_dim=64`. Plot train and val
RMSE for both. At what dimension does overfitting appear? Does a smaller embedding generalise
better on this small dataset?

**2. Activation function.** Replace `nn.ReLU()` with `nn.Tanh()` in the NCF `hidden` layers.
Does val RMSE change? Run three seeds and average to account for random variation.

**3. Cosine similarity in two-tower.** After computing `u` and `v` in `TwoTower.forward`,
L2-normalise both vectors before the dot product:

```python
u = u / (u.norm(dim=1, keepdim=True) + 1e-8)
v = v / (v.norm(dim=1, keepdim=True) + 1e-8)
```

This makes the score a cosine similarity in [-1, 1] rather than an unbounded dot product.
Does normalisation help or hurt val RMSE on this dataset? Why might it help for ranking
tasks but not for rating prediction?

> **Next:** [Sequential Recommendation](5_sequential_recommendation.ipynb) -- modelling the order in which items are consumed